# Cartographie prédictive GWS — XGBoost

Notebook Google Colab généré à partir du script Python d'origine : **`XGboost.py`**

**Dossier de données attendu sur Google Drive :** `/content/drive/MyDrive/input_data_ML`
(il doit contenir `name_files.txt` et les rasters listés dedans).

**Ordre d'exécution :** lancer les cellules de haut en bas (*Exécution → Tout exécuter*).
Le calcul est lancé par la cellule `main()` en fin de notebook.
Les résultats sont écrits dans `input_data_ML/GWS_PREDICTIONS_GEE_XGB_IDW_APIIDW_TWI_DISTRIVER` sur Drive.

Le code du script est repris tel quel. Les seules lignes ajoutées ou modifiées pour Colab
sont repérées par le commentaire `# [COLAB]`.

### Description (en-tête du script d'origine)

```text
# ============================================================
#  CARTOGRAPHIE PREDICTIVE GWS
#  (TRAIN = longue période, TEST = 5% des dates, CLIP GEE, SANS SOUS-ÉCHANTILLONNAGE)
#
#  - Modèle XGBoost sur grille GLDAS (~25 km)
#
#  - Variables explicatives (identiques au script RF) :
#       * Precipitations_IDW         → CHIRPS-IDW (journalier, issu IDW)
#       * Precipitations_API_IDW     → API-IDW (journalier, issu IDW)
#       * MODIS_LST_Clipped          → LST MODIS (journalier)
#       * MODIS_NDVI_Clipped         → NDVI MODIS (16 jours, étendu)
#       * MODIS_ET_Clipped           → ET MODIS (~8 jours, étendu)
#       * ERA5_Qsb_Clipped           → Qsb ERA5 (journalier)
#       * SMAP_RZSM_Clipped          → RZSM SMAP (journalier)
#       * SMAP_SoilM_Clipped         → SoilM SMAP (journalier)
#       * TWI (statique)
#       * Distance_To_River (statique)
#
#  - Cible :
#       * GLDAS_GWS_Clipped          → GWS (journalier)
#
#  - Features finales :
#       [P_IDW, API_IDW, LST, NDVI_ext, ET_ext, Qsb, RZSM, SoilM, TWI, DistRiver, sinDOY, cosDOY]
#
#  ➤ Périodes :
#    TRAIN pool : 2021-01-01 → 2025-06-30
#    TEST = 5% des dates (aléatoire dans la fenêtre TRAIN)
#    FORECAST : 2025-07-01 → 2025-10-30
# ============================================================
```

## A. Préparation de l'environnement Colab

### A.1 Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### A.2 Installer les dépendances

`rasterio` n'est pas préinstallé sur Colab. `xgboost` l'est en général ; la commande ne fait alors que vérifier sa présence.

In [ ]:
!pip install -q rasterio xgboost

### A.3 Imports

In [ ]:
import os
import re
import time
import bisect
from datetime import datetime, date

import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# [COLAB] Versions des bibliothèques utilisées
from importlib.metadata import version as _pkg_version
for _pkg in ('numpy', 'scikit-learn', 'rasterio', 'xgboost'):
    print(f"{_pkg:13s}: {_pkg_version(_pkg)}")

## 1. Paramètres généraux

`BASE_DIR` pointe vers le dossier `input_data_ML` de votre Drive.

In [ ]:
# ------------------------------------------------------------
# 1) Paramètres généraux
# ------------------------------------------------------------
np.random.seed(42)

# [COLAB] Dossier d'entrée sur Google Drive (remplace C:\Users\PC\Downloads\input_data_ML)
BASE_DIR = "/content/drive/MyDrive/input_data_ML"
NAME_FILE = os.path.join(BASE_DIR, "name_files.txt")

# [COLAB] Arrêt immédiat si Drive n'est pas monté ou si le dossier est mal placé
# (sinon le dossier de sortie serait créé hors de Drive et perdu en fin de session)
if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(
        f"Dossier introuvable: {BASE_DIR} -> monter Drive (cellule A.1) et vérifier "
        "que input_data_ML est bien à la racine de 'Mon Drive'."
    )
if not os.path.isfile(NAME_FILE):
    raise FileNotFoundError(f"Fichier introuvable: {NAME_FILE}")

# ✅ Dossiers IDW (comme RF)
PRECIP_IDW_DIR = os.path.join(BASE_DIR, "Precipitations_IDW")
API_IDW_DIR    = os.path.join(BASE_DIR, "Precipitations_API_IDW")

# Autres dossiers (inchangés)
LST_DIR    = os.path.join(BASE_DIR, "MODIS_LST_Clipped")
NDVI_DIR   = os.path.join(BASE_DIR, "MODIS_NDVI_Clipped")   # 16 jours
ET_DIR     = os.path.join(BASE_DIR, "MODIS_ET_Clipped")     # ~8 jours
GWS_DIR    = os.path.join(BASE_DIR, "GLDAS_GWS_Clipped")    # cible
QSB_DIR    = os.path.join(BASE_DIR, "ERA5_Qsb_Clipped")     # journalier
RZSM_DIR   = os.path.join(BASE_DIR, "SMAP_RZSM_Clipped")    # journalier
SOILM_DIR  = os.path.join(BASE_DIR, "SMAP_SoilM_Clipped")   # journalier

# ✅ Statiques (comme RF)
# [COLAB] chemins statiques reconstruits depuis BASE_DIR (au lieu de C:\...)
TWI_PATH = os.path.join(BASE_DIR, "TWI", "TWI.tif")
DIST_RIVER_PATH = os.path.join(BASE_DIR, "Distance_To_River", "Distance_To_River.tif")

OUT_DIR = os.path.join(BASE_DIR, "GWS_PREDICTIONS_GEE_XGB_IDW_APIIDW_TWI_DISTRIVER")
os.makedirs(OUT_DIR, exist_ok=True)

NODATA_VALUE = -9999.0
TEST_FRACTION = 0.05  # ✅ comme RF

# Fenêtres temporelles
TRAIN_START = date(2021, 1, 1)
TRAIN_END   = date(2025, 6, 30)

FORECAST_START = datetime(2025, 7, 1)
FORECAST_END   = datetime(2025, 10, 30)

print("=== PARAMÈTRES GÉNÉRAUX ===")
print(" BASE_DIR :", BASE_DIR)
print(" NAME_FILE :", NAME_FILE)
print(" PRECIP_IDW_DIR :", PRECIP_IDW_DIR)
print(" API_IDW_DIR    :", API_IDW_DIR)
print(" TWI_PATH :", TWI_PATH)
print(" DIST_RIVER_PATH :", DIST_RIVER_PATH)
print(" OUT_DIR :", OUT_DIR)
print(" TRAIN_START/END :", TRAIN_START, "→", TRAIN_END)
print(" TEST_FRACTION :", TEST_FRACTION)
print(" FORECAST_START/END :", FORECAST_START.date(), "→", FORECAST_END.date())
print("============================\n")

if not os.path.isfile(TWI_PATH):
    raise FileNotFoundError(f"❌ Raster TWI introuvable : {TWI_PATH}")
if not os.path.isfile(DIST_RIVER_PATH):
    raise FileNotFoundError(f"❌ Raster Distance_To_River introuvable : {DIST_RIVER_PATH}")

## 2. Dates exclues (LST manquante 2022-10-11 → 2022-10-22)

In [ ]:
# ------------------------------------------------------------
# 2) Dates exclues (LST manquante 2022-10-11 → 2022-10-22)
# ------------------------------------------------------------
EXCLUDED_DATES = set(date(2022, 10, d) for d in range(11, 23))
print("Dates exclues (LST manquante) :")
for d in sorted(EXCLUDED_DATES):
    print(" -", d)

## 3. Fonctions utilitaires

Adaptation Colab : les entrées de `name_files.txt` écrites sous Windows
(antislashs `\`, chemins absolus `C:\...\input_data_ML\...`) sont converties
automatiquement en chemins Drive par `_win_to_colab_path`.

In [ ]:
# ------------------------------------------------------------
# 3) Fonctions utilitaires
# ------------------------------------------------------------
def extract_date_from_filename(path):
    name = os.path.basename(path)
    m = re.search(r"(\d{8})", name)
    if not m:
        raise ValueError(f"Impossible d'extraire la date depuis : {name}")
    return datetime.strptime(m.group(1), "%Y%m%d").date()

def _win_to_colab_path(p: str) -> str:
    """
    [COLAB] Convertit une entrée écrite sous Windows en chemin utilisable sur Colab (Linux):
    - antislashs -> slashs
    - chemin absolu Windows (C:/.../input_data_ML/xxx) -> xxx (relatif à BASE_DIR)
    Les chemins déjà valides (relatifs, ou absolus sous BASE_DIR) sont inchangés.
    """
    p = str(p).strip().replace("\\", "/")
    if re.match(r"^[A-Za-z]:/", p):
        root_name = os.path.basename(os.path.normpath(BASE_DIR)).lower()
        parts = p.split("/")
        low = [x.lower() for x in parts]
        if root_name in low:
            rest = parts[low.index(root_name) + 1:]
            p = "/".join(rest) if rest else "."
    return p


def parse_name_file(txt_path):
    sections = {}
    current = None
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("[") and line.endswith("]"):
                current = line[1:-1].strip()
                sections[current] = []
            else:
                if current is None:
                    continue
                sections[current].append(_win_to_colab_path(line))  # [COLAB] chemin Windows -> Colab
    return sections

def build_date_to_path_dict_from_section(sections, section_name):
    if section_name not in sections:
        raise RuntimeError(f"Section '{section_name}' absente de {NAME_FILE}")
    d = {}
    for rel_path in sections[section_name]:
        abs_path = os.path.normpath(os.path.join(BASE_DIR, rel_path))
        if not os.path.isfile(abs_path):
            print(f"⚠️ Fichier introuvable (ignoré) : {abs_path}")
            continue
        try:
            date_key = extract_date_from_filename(abs_path)
            d[date_key] = abs_path
        except ValueError as e:
            print("⚠️", e)
    return d

def read_raster_raw(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile
        nodata = src.nodata
    return arr, profile, nodata

def to_float_with_nan(arr, nodata):
    arr = arr.astype("float32")
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)
    return arr

def reproject_to_template(
    src_arr,
    src_profile,
    template_profile,
    src_nodata=None,
    dst_nodata=np.nan,
    resampling=Resampling.average,
):
    dst_arr = np.full(
        (template_profile["height"], template_profile["width"]),
        dst_nodata,
        dtype="float32",
    )
    kwargs = dict(
        source=src_arr,
        destination=dst_arr,
        src_transform=src_profile["transform"],
        src_crs=src_profile["crs"],
        dst_transform=template_profile["transform"],
        dst_crs=template_profile["crs"],
        dst_nodata=dst_nodata,
        resampling=resampling,
    )
    if src_nodata is not None:
        kwargs["src_nodata"] = src_nodata
    reproject(**kwargs)
    return dst_arr

def fill_nan_with_mean(arr):
    if np.all(np.isnan(arr)):
        return arr
    m = np.nanmean(arr)
    return np.where(np.isnan(arr), m, arr)

def build_prev_date_support(date_dict):
    dates_sorted = sorted(date_dict.keys())
    def get_prev(target_date):
        if not dates_sorted:
            return None
        idx = bisect.bisect_right(dates_sorted, target_date) - 1
        if idx < 0:
            return None
        return dates_sorted[idx]
    return get_prev

# --- Masque "Regions" reconstruit à partir des rasters ---
REGION_MASK = None

def update_region_mask(p_idw_res, lst_res, ndvi_res, template_profile):
    global REGION_MASK
    base_mask = ~(np.isnan(p_idw_res) & np.isnan(lst_res) & np.isnan(ndvi_res))
    if REGION_MASK is None:
        REGION_MASK = base_mask
        print("\n🗺️ Masque 'Regions' initial construit à partir des rasters (1er jour).")
    else:
        REGION_MASK = REGION_MASK | base_mask

def get_region_mask(template_profile):
    global REGION_MASK
    if REGION_MASK is None:
        print("⚠️ REGION_MASK non initialisé, on prend toute la grille.")
        REGION_MASK = np.ones((template_profile["height"], template_profile["width"]), dtype=bool)
    return REGION_MASK

### 3.1 Rasters statiques sur la grille template (cache)

In [ ]:
# ------------------------------------------------------------
# 3.1) Rasters statiques sur la grille template (cache)
# ------------------------------------------------------------
TWI_ON_TEMPLATE = None
DIST_ON_TEMPLATE = None

def ensure_static_on_template(template_profile):
    global TWI_ON_TEMPLATE, DIST_ON_TEMPLATE

    if TWI_ON_TEMPLATE is None:
        twi_raw, twi_prof, twi_nodata = read_raster_raw(TWI_PATH)
        twi_res = reproject_to_template(
            twi_raw, twi_prof, template_profile,
            src_nodata=twi_nodata, dst_nodata=np.nan,
            resampling=Resampling.bilinear
        )
        TWI_ON_TEMPLATE = fill_nan_with_mean(twi_res).astype("float32")
        print("\n✅ TWI reprojeté sur la grille GLDAS (template).")

    if DIST_ON_TEMPLATE is None:
        dist_raw, dist_prof, dist_nodata = read_raster_raw(DIST_RIVER_PATH)
        dist_res = reproject_to_template(
            dist_raw, dist_prof, template_profile,
            src_nodata=dist_nodata, dst_nodata=np.nan,
            resampling=Resampling.bilinear
        )
        DIST_ON_TEMPLATE = fill_nan_with_mean(dist_res).astype("float32")
        print("✅ Distance_To_River reprojetée sur la grille GLDAS (template).")

    return TWI_ON_TEMPLATE, DIST_ON_TEMPLATE

### Contrôle des entrées de `name_files.txt` (ajout Colab)

Vérifie, avant le calcul long, que chaque entrée du fichier se résout bien en fichier présent sur Drive.
Une section en ❌ signale un chemin à corriger. Lecture seule ; peut prendre une à deux minutes sur Drive.

In [ ]:
# [COLAB] Contrôle rapide : les entrées de name_files.txt se résolvent-elles sur Drive ?
_sections_check = parse_name_file(NAME_FILE)
print(f"{len(_sections_check)} sections lues dans {os.path.basename(NAME_FILE)}\n")

for _sec, _entries in _sections_check.items():
    _missing = [_e for _e in _entries
                if not os.path.isfile(os.path.normpath(os.path.join(BASE_DIR, _e)))]
    _n_ok = len(_entries) - len(_missing)
    _flag = "✅" if (_n_ok > 0 and not _missing) else ("⚠️" if _n_ok > 0 else "❌")
    print(f"{_flag} [{_sec}] {len(_entries)} entrées -> {_n_ok} fichiers | introuvables: {len(_missing)}")
    for _e in _missing[:3]:
        print(f"      ex. introuvable: {os.path.normpath(os.path.join(BASE_DIR, _e))}")

## Pipeline principal — définition de `main()`

Cette cellule ne fait que définir la fonction ; le calcul est lancé dans la cellule suivante.

In [ ]:
# ============================================================
# MAIN
# ============================================================
def main():
    # ------------------------------------------------------------
    # 4) Indexation des rasters par date (via name_files.txt)
    # ------------------------------------------------------------
    print("\n=== INDEXATION DES FICHIERS PAR DATE (via name_files.txt) ===")
    sections = parse_name_file(NAME_FILE)

    # ✅ Sections IDW obligatoires
    precip_dict = build_date_to_path_dict_from_section(sections, "Precipitations_IDW")
    api_dict    = build_date_to_path_dict_from_section(sections, "Precipitations_API_IDW")

    lst_dict    = build_date_to_path_dict_from_section(sections, "MODIS_LST_Clipped")
    ndvi_dict   = build_date_to_path_dict_from_section(sections, "MODIS_NDVI_Clipped")
    et_dict     = build_date_to_path_dict_from_section(sections, "MODIS_ET_Clipped")
    gws_dict    = build_date_to_path_dict_from_section(sections, "GLDAS_GWS_Clipped")
    qsb_dict    = build_date_to_path_dict_from_section(sections, "ERA5_Qsb_Clipped")
    rzsm_dict   = build_date_to_path_dict_from_section(sections, "SMAP_RZSM_Clipped")
    soilm_dict  = build_date_to_path_dict_from_section(sections, "SMAP_SoilM_Clipped")

    print("Nb dates Precip_IDW :", len(precip_dict))
    print("Nb dates API_IDW    :", len(api_dict))
    print("Nb dates LST        :", len(lst_dict))
    print("Nb dates NDVI       :", len(ndvi_dict))
    print("Nb dates ET         :", len(et_dict))
    print("Nb dates GWS        :", len(gws_dict))
    print("Nb dates Qsb        :", len(qsb_dict))
    print("Nb dates RZSM       :", len(rzsm_dict))
    print("Nb dates SoilM      :", len(soilm_dict))

    if len(precip_dict) == 0:
        raise RuntimeError("⚠️ precip_dict est vide. Vérifie la section [Precipitations_IDW] dans name_files.txt.")
    if len(api_dict) == 0:
        raise RuntimeError("⚠️ api_dict est vide. Vérifie la section [Precipitations_API_IDW] dans name_files.txt.")

    get_ndvi_date_for = build_prev_date_support(ndvi_dict)
    get_et_date_for   = build_prev_date_support(et_dict)

    # ------------------------------------------------------------
    # 5) Dates TRAIN / TEST (5%)
    # ------------------------------------------------------------
    print("\n=== CONSTRUCTION DES DATES TRAIN / TEST (5% des dates pour TEST) ===")

    # Intersection journaliers (inclut IDW)
    all_common_dates = sorted(
        set(precip_dict.keys())
        & set(api_dict.keys())
        & set(lst_dict.keys())
        & set(gws_dict.keys())
        & set(qsb_dict.keys())
        & set(rzsm_dict.keys())
        & set(soilm_dict.keys())
    )
    print("Nombre total de dates communes :", len(all_common_dates))
    if not all_common_dates:
        raise RuntimeError("⚠️ Aucune date commune entre les variables journalières (incluant IDW + API_IDW).")

    date_candidates = [
        d for d in all_common_dates
        if (TRAIN_START <= d <= TRAIN_END) and (d not in EXCLUDED_DATES)
    ]
    if len(date_candidates) == 0:
        raise RuntimeError("⚠️ Aucune date candidate dans la fenêtre TRAIN (après exclusion).")

    train_dates, test_dates = train_test_split(
        date_candidates,
        test_size=TEST_FRACTION,
        random_state=42,
        shuffle=True,
    )
    train_dates = sorted(train_dates)
    test_dates  = sorted(test_dates)

    print(f"\nFenêtre TRAIN : {TRAIN_START} → {TRAIN_END}")
    print(f" → Dates train : {len(train_dates)}")
    print(f"\nTEST = {int(TEST_FRACTION*100)}%")
    print(f" → Dates test  : {len(test_dates)}")

    # ------------------------------------------------------------
    # 6) Build dataset X/y (pixels)
    # ------------------------------------------------------------
    print("\n=== CONSTRUCTION DU DATASET (toutes les dates TRAIN + TEST) ===")

    X_train_list, y_train_list = [], []
    X_test_list,  y_test_list  = [], []

    template_profile = None

    def build_Xy_for_date(date_key, template_profile, role="train"):
        # NDVI/ET étendus
        ndvi_date = get_ndvi_date_for(date_key)
        if ndvi_date is None:
            print(f"⏭️ [{role}] {date_key} : aucun NDVI <= date, ignoré.")
            return None, None, template_profile

        et_date = get_et_date_for(date_key)
        if et_date is None:
            print(f"⏭️ [{role}] {date_key} : aucun ET <= date, ignoré.")
            return None, None, template_profile

        # GWS
        if date_key not in gws_dict:
            print(f"⏭️ [{role}] {date_key} : GWS manquant, ignoré.")
            return None, None, template_profile

        gws_raw, gws_prof, gws_nodata = read_raster_raw(gws_dict[date_key])
        gws_arr = to_float_with_nan(gws_raw, gws_nodata)

        if template_profile is None:
            template_profile = gws_prof
            print("\nGrille GLDAS (template) :")
            print(" - CRS :", template_profile["crs"])
            print(" - Taille:", template_profile["width"], "x", template_profile["height"])
            print(" - Résolution approx (deg) :", template_profile["transform"][0])
            ensure_static_on_template(template_profile)

        # Vérif journaliers
        if (
            date_key not in precip_dict
            or date_key not in api_dict
            or date_key not in lst_dict
            or date_key not in qsb_dict
            or date_key not in rzsm_dict
            or date_key not in soilm_dict
        ):
            print(f"⏭️ [{role}] {date_key} : variable journalière manquante, ignoré.")
            return None, None, template_profile

        # Lecture entrées
        p_raw,   p_prof,   p_nodata   = read_raster_raw(precip_dict[date_key])
        api_raw, api_prof, api_nodata = read_raster_raw(api_dict[date_key])
        lst_raw, lst_prof, lst_nodata = read_raster_raw(lst_dict[date_key])

        ndvi_raw, ndvi_prof, ndvi_nodata = read_raster_raw(ndvi_dict[ndvi_date])
        et_raw,   et_prof,   et_nodata   = read_raster_raw(et_dict[et_date])

        qsb_raw, qsb_prof, qsb_nodata = read_raster_raw(qsb_dict[date_key])
        rzsm_raw, rzsm_prof, rzsm_nodata = read_raster_raw(rzsm_dict[date_key])
        soilm_raw, soilm_prof, soilm_nodata = read_raster_raw(soilm_dict[date_key])

        # Rééchantillonnage sur GLDAS
        p_res   = reproject_to_template(p_raw,   p_prof,   template_profile, src_nodata=p_nodata,   dst_nodata=np.nan)
        api_res = reproject_to_template(api_raw, api_prof, template_profile, src_nodata=api_nodata, dst_nodata=np.nan)
        lst_res = reproject_to_template(lst_raw, lst_prof, template_profile, src_nodata=lst_nodata, dst_nodata=np.nan)

        ndvi_res = reproject_to_template(ndvi_raw, ndvi_prof, template_profile, src_nodata=ndvi_nodata, dst_nodata=np.nan)
        et_res   = reproject_to_template(et_raw,   et_prof,   template_profile, src_nodata=et_nodata,   dst_nodata=np.nan)

        qsb_res  = reproject_to_template(qsb_raw,  qsb_prof,  template_profile, src_nodata=qsb_nodata,  dst_nodata=np.nan)
        rzsm_res = reproject_to_template(rzsm_raw, rzsm_prof, template_profile, src_nodata=rzsm_nodata, dst_nodata=np.nan)
        soilm_res= reproject_to_template(soilm_raw,soilm_prof,template_profile, src_nodata=soilm_nodata,dst_nodata=np.nan)

        # Masque Regions (basé sur P_IDW/LST/NDVI)
        update_region_mask(p_res, lst_res, ndvi_res, template_profile)
        region_mask = get_region_mask(template_profile)

        # Fill NaN (pour cartes sans trous)
        p_f     = fill_nan_with_mean(p_res)
        api_f   = fill_nan_with_mean(api_res)
        lst_f   = fill_nan_with_mean(lst_res)
        ndvi_f  = fill_nan_with_mean(ndvi_res)
        et_f    = fill_nan_with_mean(et_res)
        qsb_f   = fill_nan_with_mean(qsb_res)
        rzsm_f  = fill_nan_with_mean(rzsm_res)
        soilm_f = fill_nan_with_mean(soilm_res)

        twi_2d, dist_2d = ensure_static_on_template(template_profile)

        # Pixels valides (GWS défini + Regions)
        mask_valid = (~np.isnan(gws_arr)) & region_mask
        if not np.any(mask_valid):
            print(f"⏭️ [{role}] {date_key} : aucun pixel valide, ignoré.")
            return None, None, template_profile

        # DOY sin/cos
        doy = date_key.timetuple().tm_yday
        sin_doy = np.sin(2 * np.pi * doy / 365.0)
        cos_doy = np.cos(2 * np.pi * doy / 365.0)

        n_pixels = np.count_nonzero(mask_valid)
        sin_plane = np.full(n_pixels, sin_doy, dtype="float32")
        cos_plane = np.full(n_pixels, cos_doy, dtype="float32")

        # ✅ Features identiques au RF
        X_day = np.stack(
            [
                p_f[mask_valid],
                api_f[mask_valid],
                lst_f[mask_valid],
                ndvi_f[mask_valid],
                et_f[mask_valid],
                qsb_f[mask_valid],
                rzsm_f[mask_valid],
                soilm_f[mask_valid],
                twi_2d[mask_valid],
                dist_2d[mask_valid],
                sin_plane,
                cos_plane,
            ],
            axis=1,
        )
        y_day = gws_arr[mask_valid]

        # Filtrage final (sécurité NaN)
        row_mask = (~np.isnan(y_day)) & (~np.isnan(X_day).any(axis=1))
        if not np.any(row_mask):
            print(f"⏭️ [{role}] {date_key} : pixels invalides (NaN features), ignoré.")
            return None, None, template_profile

        X_day = X_day[row_mask]
        y_day = y_day[row_mask]

        print(f" ✔️ [{role}] {date_key} : {X_day.shape[0]} pixels utilisés.")
        return X_day, y_day, template_profile

    # TRAIN
    for d in train_dates:
        X_day, y_day, template_profile = build_Xy_for_date(d, template_profile, role="train")
        if X_day is None:
            continue
        X_train_list.append(X_day)
        y_train_list.append(y_day)

    # TEST
    for d in test_dates:
        X_day, y_day, template_profile = build_Xy_for_date(d, template_profile, role="test")
        if X_day is None:
            continue
        X_test_list.append(X_day)
        y_test_list.append(y_day)

    if len(X_train_list) == 0:
        raise RuntimeError("⚠️ Aucun pixel d'entraînement valide n'a été trouvé.")
    if len(X_test_list) == 0:
        raise RuntimeError("⚠️ Aucun pixel de test valide n'a été trouvé.")

    X_train = np.vstack(X_train_list).astype("float32")
    y_train = np.concatenate(y_train_list).astype("float32")
    X_test  = np.vstack(X_test_list).astype("float32")
    y_test  = np.concatenate(y_test_list).astype("float32")

    print(f"\nPixels TRAIN : {X_train.shape[0]}")
    print(f"Pixels TEST  : {X_test.shape[0]}")
    print("✅ Nombre de features :", X_train.shape[1])  # doit être 12

    # ------------------------------------------------------------
    # 7) Modèle XGBoost
    # ------------------------------------------------------------
    print("\n=== ENTRAÎNEMENT DU MODÈLE XGBOOST ===")

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train,
        test_size=0.2, random_state=42, shuffle=True
    )
    print(f"Taille X_tr : {X_tr.shape}, X_val : {X_val.shape}")

    xgb_model = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=2000,
        learning_rate=0.01,
        max_depth=8,
        min_child_weight=5,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_alpha=0.1,
        reg_lambda=0.3,
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
        eval_metric="rmse",
    )

    print("\n🔄 Entraînement XGBoost (train / valid)...")
    t0 = time.time()
    xgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_tr, y_tr), (X_val, y_val)],
        verbose=False,
    )
    t1 = time.time()
    training_time_ms = (t1 - t0) * 1000.0

    # Metrics pixel-wise
    y_tr_pred   = xgb_model.predict(X_tr)
    y_val_pred  = xgb_model.predict(X_val)
    y_test_pred = xgb_model.predict(X_test)

    r2_tr   = r2_score(y_tr, y_tr_pred)
    rmse_tr = np.sqrt(mean_squared_error(y_tr, y_tr_pred))
    mae_tr  = mean_absolute_error(y_tr, y_tr_pred)

    r2_val   = r2_score(y_val, y_val_pred)
    rmse_val = np.sqrt(mean_squared_error(y_val, y_val_pred))
    mae_val  = mean_absolute_error(y_val, y_val_pred)

    r2_test   = r2_score(y_test, y_test_pred)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
    mae_test  = mean_absolute_error(y_test, y_test_pred)

    print("\n╔═════════ MODÈLE XGBOOST ═════════╗")
    print(f" ➤ Temps d'entraînement : {training_time_ms:.2f} ms")
    print(" --- TRAIN (pixels) ---")
    print(f"    R² : {r2_tr:.4f} | MAE : {mae_tr:.4f} | RMSE : {rmse_tr:.4f}")
    print(" --- VALIDATION (pixels) ---")
    print(f"    R² : {r2_val:.4f} | MAE : {mae_val:.4f} | RMSE : {rmse_val:.4f}")
    print(" --- TEST (dates 5%) (pixels) ---")
    print(f"    R² : {r2_test:.4f} | MAE : {mae_test:.4f} | RMSE : {rmse_test:.4f}")
    print("╚══════════════════════════════════╝")

    # Refit final sur tout X_train (comme logique RF avant carto/forecast)
    print("\n🔁 Ré-entrainement final sur TOUT X_train (cartographie + forecast)...")
    xgb_model.fit(X_train, y_train, verbose=False)
    print("   ✔️ xgb_model prêt.")

    # ------------------------------------------------------------
    # 8) Évaluation cartographique (dates TEST) + sauvegarde GeoTIFF
    # ------------------------------------------------------------
    print("\n=== ÉVALUATION CARTOGRAPHIQUE SUR TOUTES LES DATES TEST (5%) ===")
    region_mask = get_region_mask(template_profile)

    all_true, all_pred = [], []

    out_profile_test = template_profile.copy()
    out_profile_test.update(dtype="float32", count=1, nodata=NODATA_VALUE)

    for date_key in test_dates:
        print(f"\n🧪 Test cartographique pour la date : {date_key}")

        if date_key in EXCLUDED_DATES:
            print(" ⏭️ Date exclue, ignorée.")
            continue

        ndvi_date = get_ndvi_date_for(date_key)
        et_date   = get_et_date_for(date_key)
        if (ndvi_date is None) or (et_date is None):
            print(" ⏭️ NDVI/ET indisponible, ignorée.")
            continue

        if date_key not in gws_dict:
            print(" ⏭️ GWS manquant, ignorée.")
            continue

        # vérité
        gws_raw, _, gws_nodata = read_raster_raw(gws_dict[date_key])
        gws_arr = to_float_with_nan(gws_raw, gws_nodata)

        # check journaliers
        if (date_key not in precip_dict or date_key not in api_dict or date_key not in lst_dict
            or date_key not in qsb_dict or date_key not in rzsm_dict or date_key not in soilm_dict):
            print(" ⏭️ Variable journalière manquante, ignorée.")
            continue

        # lecture
        p_raw, p_prof, p_nodata = read_raster_raw(precip_dict[date_key])
        api_raw, api_prof, api_nodata = read_raster_raw(api_dict[date_key])
        lst_raw, lst_prof, lst_nodata = read_raster_raw(lst_dict[date_key])

        ndvi_raw, ndvi_prof, ndvi_nodata = read_raster_raw(ndvi_dict[ndvi_date])
        et_raw, et_prof, et_nodata = read_raster_raw(et_dict[et_date])

        qsb_raw, qsb_prof, qsb_nodata = read_raster_raw(qsb_dict[date_key])
        rzsm_raw, rzsm_prof, rzsm_nodata = read_raster_raw(rzsm_dict[date_key])
        soilm_raw, soilm_prof, soilm_nodata = read_raster_raw(soilm_dict[date_key])

        # reprojection
        p_res   = reproject_to_template(p_raw, p_prof, template_profile, src_nodata=p_nodata, dst_nodata=np.nan)
        api_res = reproject_to_template(api_raw, api_prof, template_profile, src_nodata=api_nodata, dst_nodata=np.nan)
        lst_res = reproject_to_template(lst_raw, lst_prof, template_profile, src_nodata=lst_nodata, dst_nodata=np.nan)

        ndvi_res = reproject_to_template(ndvi_raw, ndvi_prof, template_profile, src_nodata=ndvi_nodata, dst_nodata=np.nan)
        et_res   = reproject_to_template(et_raw,   et_prof,   template_profile, src_nodata=et_nodata,   dst_nodata=np.nan)

        qsb_res  = reproject_to_template(qsb_raw,  qsb_prof,  template_profile, src_nodata=qsb_nodata,  dst_nodata=np.nan)
        rzsm_res = reproject_to_template(rzsm_raw, rzsm_prof, template_profile, src_nodata=rzsm_nodata, dst_nodata=np.nan)
        soilm_res= reproject_to_template(soilm_raw,soilm_prof,template_profile, src_nodata=soilm_nodata,dst_nodata=np.nan)

        # fill NaN
        p_f     = fill_nan_with_mean(p_res)
        api_f   = fill_nan_with_mean(api_res)
        lst_f   = fill_nan_with_mean(lst_res)
        ndvi_f  = fill_nan_with_mean(ndvi_res)
        et_f    = fill_nan_with_mean(et_res)
        qsb_f   = fill_nan_with_mean(qsb_res)
        rzsm_f  = fill_nan_with_mean(rzsm_res)
        soilm_f = fill_nan_with_mean(soilm_res)

        twi_2d, dist_2d = ensure_static_on_template(template_profile)

        # DOY
        doy = date_key.timetuple().tm_yday
        sin_doy = np.sin(2 * np.pi * doy / 365.0)
        cos_doy = np.cos(2 * np.pi * doy / 365.0)

        n_pixels_full = p_f.size
        sin_plane_full = np.full(n_pixels_full, sin_doy, dtype="float32")
        cos_plane_full = np.full(n_pixels_full, cos_doy, dtype="float32")

        X_all = np.stack(
            [
                p_f.ravel(),
                api_f.ravel(),
                lst_f.ravel(),
                ndvi_f.ravel(),
                et_f.ravel(),
                qsb_f.ravel(),
                rzsm_f.ravel(),
                soilm_f.ravel(),
                twi_2d.ravel(),
                dist_2d.ravel(),
                sin_plane_full,
                cos_plane_full,
            ],
            axis=1,
        )

        if np.isnan(X_all).any():
            print(" ⏭️ X_all contient des NaN, ignorée.")
            continue

        y_pred_all = xgb_model.predict(X_all).astype("float32")
        pred_full = y_pred_all.reshape(template_profile["height"], template_profile["width"])

        pred_clip = pred_full.copy()
        pred_clip[~region_mask] = NODATA_VALUE

        mask_gws = (~np.isnan(gws_arr)) & region_mask
        if not np.any(mask_gws):
            print(" ⚠️ Aucun pixel GWS valide, ignorée.")
            continue

        y_true = gws_arr[mask_gws]
        y_pred = pred_clip[mask_gws]

        r2_d = r2_score(y_true, y_pred)
        mae_d = mean_absolute_error(y_true, y_pred)
        rmse_d = np.sqrt(mean_squared_error(y_true, y_pred))

        print(f" ➤ R² (carte clipée) : {r2_d:.4f}")
        print(f" ➤ MAE (carte clipée) : {mae_d:.4f}")
        print(f" ➤ RMSE (carte clipée) : {rmse_d:.4f}")

        all_true.append(y_true)
        all_pred.append(y_pred)

        out_name_test = os.path.join(OUT_DIR, f"GWS_TEST_PRED_{date_key.strftime('%Y%m%d')}.tif")
        with rasterio.open(out_name_test, "w", **out_profile_test) as dst:
            dst.write(pred_clip, 1)
        print(f" ✔️ Carte test prédite : {out_name_test}")

    if len(all_true) > 0:
        all_true_flat = np.concatenate(all_true)
        all_pred_flat = np.concatenate(all_pred)
        r2_global = r2_score(all_true_flat, all_pred_flat)
        mae_global = mean_absolute_error(all_true_flat, all_pred_flat)
        rmse_global = np.sqrt(mean_squared_error(all_true_flat, all_pred_flat))

        print("\n📊 MÉTRIQUES GLOBALES (toutes dates TEST, cartes clipées)")
        print(f" ➤ R² global : {r2_global:.4f}")
        print(f" ➤ MAE global : {mae_global:.4f}")
        print(f" ➤ RMSE global: {rmse_global:.4f}")
    else:
        print("\n⚠️ Aucune métrique globale calculable (aucun pixel valide).")

    # ------------------------------------------------------------
    # 9) Phase de PRÉDICTION CARTOGRAPHIQUE (FORECAST)
    # ------------------------------------------------------------
    print("\n=== PHASE DE PRÉDICTION CARTOGRAPHIQUE (SANS TROUS, CLIPÉE REGIONS) ===")

    out_profile = template_profile.copy()
    out_profile.update(dtype="float32", count=1, nodata=NODATA_VALUE)

    all_forecast_dates = sorted(
        set(precip_dict.keys())
        & set(api_dict.keys())
        & set(lst_dict.keys())
        & set(qsb_dict.keys())
        & set(rzsm_dict.keys())
        & set(soilm_dict.keys())
    )

    user_start = FORECAST_START.date()
    user_end = FORECAST_END.date()

    if all_forecast_dates:
        forecast_start_date = max(user_start, min(all_forecast_dates))
        forecast_end_date = min(user_end, max(all_forecast_dates))
    else:
        forecast_start_date = user_start
        forecast_end_date = user_end

    forecast_dates = [
        d for d in all_forecast_dates
        if (forecast_start_date <= d <= forecast_end_date) and (d not in EXCLUDED_DATES)
    ]

    print("Fenêtre demandée :", user_start, "→", user_end)
    print("Fenêtre utilisée :", forecast_start_date, "→", forecast_end_date)
    print("Nb dates forecast éligibles :", len(forecast_dates))

    region_mask = get_region_mask(template_profile)

    for date_key in forecast_dates:
        if date_key in EXCLUDED_DATES:
            continue

        ndvi_date = get_ndvi_date_for(date_key)
        et_date   = get_et_date_for(date_key)
        if (ndvi_date is None) or (et_date is None):
            continue

        # lecture
        p_raw, p_prof, p_nodata = read_raster_raw(precip_dict[date_key])
        api_raw, api_prof, api_nodata = read_raster_raw(api_dict[date_key])
        lst_raw, lst_prof, lst_nodata = read_raster_raw(lst_dict[date_key])

        ndvi_raw, ndvi_prof, ndvi_nodata = read_raster_raw(ndvi_dict[ndvi_date])
        et_raw, et_prof, et_nodata = read_raster_raw(et_dict[et_date])

        qsb_raw, qsb_prof, qsb_nodata = read_raster_raw(qsb_dict[date_key])
        rzsm_raw, rzsm_prof, rzsm_nodata = read_raster_raw(rzsm_dict[date_key])
        soilm_raw, soilm_prof, soilm_nodata = read_raster_raw(soilm_dict[date_key])

        # reprojection
        p_res   = reproject_to_template(p_raw, p_prof, template_profile, src_nodata=p_nodata, dst_nodata=np.nan)
        api_res = reproject_to_template(api_raw, api_prof, template_profile, src_nodata=api_nodata, dst_nodata=np.nan)
        lst_res = reproject_to_template(lst_raw, lst_prof, template_profile, src_nodata=lst_nodata, dst_nodata=np.nan)

        ndvi_res = reproject_to_template(ndvi_raw, ndvi_prof, template_profile, src_nodata=ndvi_nodata, dst_nodata=np.nan)
        et_res   = reproject_to_template(et_raw,   et_prof,   template_profile, src_nodata=et_nodata,   dst_nodata=np.nan)

        qsb_res  = reproject_to_template(qsb_raw,  qsb_prof,  template_profile, src_nodata=qsb_nodata,  dst_nodata=np.nan)
        rzsm_res = reproject_to_template(rzsm_raw, rzsm_prof, template_profile, src_nodata=rzsm_nodata, dst_nodata=np.nan)
        soilm_res= reproject_to_template(soilm_raw,soilm_prof,template_profile, src_nodata=soilm_nodata,dst_nodata=np.nan)

        # fill NaN
        p_f     = fill_nan_with_mean(p_res)
        api_f   = fill_nan_with_mean(api_res)
        lst_f   = fill_nan_with_mean(lst_res)
        ndvi_f  = fill_nan_with_mean(ndvi_res)
        et_f    = fill_nan_with_mean(et_res)
        qsb_f   = fill_nan_with_mean(qsb_res)
        rzsm_f  = fill_nan_with_mean(rzsm_res)
        soilm_f = fill_nan_with_mean(soilm_res)

        twi_2d, dist_2d = ensure_static_on_template(template_profile)

        # DOY
        doy = date_key.timetuple().tm_yday
        sin_doy = np.sin(2 * np.pi * doy / 365.0)
        cos_doy = np.cos(2 * np.pi * doy / 365.0)

        n_pixels_full = p_f.size
        sin_plane_full = np.full(n_pixels_full, sin_doy, dtype="float32")
        cos_plane_full = np.full(n_pixels_full, cos_doy, dtype="float32")

        X_all = np.stack(
            [
                p_f.ravel(),
                api_f.ravel(),
                lst_f.ravel(),
                ndvi_f.ravel(),
                et_f.ravel(),
                qsb_f.ravel(),
                rzsm_f.ravel(),
                soilm_f.ravel(),
                twi_2d.ravel(),
                dist_2d.ravel(),
                sin_plane_full,
                cos_plane_full,
            ],
            axis=1,
        )

        if np.isnan(X_all).any():
            continue

        y_pred_all = xgb_model.predict(X_all).astype("float32")
        pred_full = y_pred_all.reshape(template_profile["height"], template_profile["width"])

        pred_clip = pred_full.copy()
        pred_clip[~region_mask] = NODATA_VALUE

        out_name = os.path.join(OUT_DIR, f"GWS_PRED_{date_key.strftime('%Y%m%d')}.tif")
        with rasterio.open(out_name, "w", **out_profile) as dst:
            dst.write(pred_clip, 1)

        print(f" ✔️ Forecast prédite : {out_name}")

    tif_files = [f for f in os.listdir(OUT_DIR) if f.lower().endswith(".tif")]
    print(f"\n📂 Nombre total de cartes .tif dans {OUT_DIR} : {len(tif_files)}")

## Exécution

Lance toute la chaîne : indexation → dataset → entraînement → évaluation → cartes prédictives.

In [ ]:
main()

## Récapitulatif des sorties (ajout Colab)

In [ ]:
# [COLAB] Nombre de fichiers présents dans le dossier de sortie, par extension
from collections import Counter

_count = Counter(os.path.splitext(_f)[1].lower() or "(sans extension)" for _f in os.listdir(OUT_DIR))
for _ext, _n in sorted(_count.items()):
    print(f"{_n:6d} fichier(s) {_ext}")
print("Dossier de sortie :", OUT_DIR)